# ✏️ 02 — Redact & Verify
Redacts PII found in a file and runs the leakage verifier to confirm no PII survives.

In [ ]:
import sys, pathlib, tempfile
ROOT = pathlib.Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import yaml
config = yaml.safe_load(open(ROOT / 'config.yaml'))

In [ ]:
# Change this to any file you want to process
FILE_PATH = ROOT / 'synthetic' / 'employee_records.docx'

In [ ]:
from extract.router import route
from detect.merge import detect_chunk
from redact.mapping import PlaceholderMapping
from redact.redactor import redact_chunks
from verify.leakage import check_leakage

chunks = route(str(FILE_PATH), config)
entities = [e for chunk in chunks for e in detect_chunk(chunk, config)]

out_dir = tempfile.mkdtemp(prefix='pii_nb_')
mapping = PlaceholderMapping(str(FILE_PATH), out_dir)
redacted = redact_chunks(chunks, entities, mapping, config)
mapping.save()

leak = check_leakage(redacted, config, known_values=mapping)
print('Downstream safe:', leak['downstream_safe'])
print('Leaks found:', leak['leaks_found'])
print('Residues:', leak.get('residues', []))

In [ ]:
# Show first 5 redacted chunks
for c in redacted[:5]:
    if not c.is_blocked():
        print(f'--- {c.location} ---')
        print(c.text[:300])
        print()